# E3 comparison — 20 random videos from each external dataset

Five local pools: RWF-2000, XD-Violence, CUHK Avenue testing videos, CUVA, and AVA. 20 files per dataset, uniformly sampled without replacement using fixed per-dataset seeds derived from 20261116. No external UCF accuracy is computed; native labels are not mapped to the UCF taxonomy.

Model: final D-76 E3, fixed three-arm probability average. Original unsaved logistic estimators are reconstructed from the registered training-only nested-C recipe and checked against saved cached final probabilities. No new backbone training, fine-tuning, fusion search or threshold tuning.

**Inference policy:** complete videos, registered stride five, 224×224 preprocessing, all image-frame features averaged; V-JEPA uses non-overlapping 16-frame clips with trailing partial clips dropped. This is denser than historical study-frame sampling, so this deployment replay is not a replacement benchmark result.

GPU inference runs in the frozen batch workflow; the notebook reads its immutable receipts. It does not start expensive work on the shared head node.

In [1]:
from pathlib import Path
import json, csv, html
from IPython.display import HTML, display
RUN = Path('<repo>/results/demo/e3-comparison-20261116')
GROUP = 'external'
config = json.loads((RUN / 'config.json').read_text())
print(config['id'], config['model'])
print('Sampling:', config['inference_policy'])

E3-ALL-UCF-EXTERNAL-20261116 D-76 E3; equal mean of L_study, B+S_study, B+VJ logistic probabilities
Sampling: Deployment replay: all registered stride-five UCF blob frames; all stride-five decoded frames of each external video, without duration/frame caps. All image/clip features are averaged per video. L/S and some VJ sampling are denser than the original historical study caches, so deployment replay is not an exact repeat of the old evaluation protocol.


## Run provenance and completeness

Configuration, source/weight digests, package versions and append-only attempts are stored beside the answers. If execution has not completed, this notebook stops here rather than showing invented findings.

In [2]:
summary_path = RUN / (GROUP + '_summary.json')
if not summary_path.exists():
    raise RuntimeError('E3 run has not completed; no result summary exists yet.')
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))
assert summary['complete'], 'Run incomplete: inspect retained failures and attempt stream.'

{
  "group": "external",
  "planned": 100,
  "recorded": 100,
  "successful": 99,
  "failed": 1,
  "complete": true,
  "failures": [
    {
      "key": "afb7147073eea02316954848",
      "error": "ValueError('Fewer than 16 selected frames: no registered complete V-JEPA clip')"
    }
  ],
  "prediction_counts": {
    "Assault": 7,
    "Normal_Videos_event": 34,
    "RoadAccidents": 1,
    "Abuse": 18,
    "Shoplifting": 4,
    "Explosion": 15,
    "Fighting": 12,
    "Arrest": 3,
    "Vandalism": 1,
    "Stealing": 1,
    "Robbery": 1,
    "Arson": 2
  },
  "by_dataset": {
    "RWF-2000": {
      "successful": 20,
      "mean_max_probability": 0.4471128589020939,
      "prediction_counts": {
        "Normal_Videos_event": 3,
        "Shoplifting": 3,
        "Fighting": 7,
        "Arrest": 1,
        "Abuse": 4,
        "Assault": 2
      },
      "ucf_accuracy": null,
      "reason": "No compatible registered UCF ground truth; native labels retained as metadata."
    },
    "XD-Violenc

## Per-video predictions

Scores below are model probabilities, not calibrated correctness probabilities. Failures remain visible and are not silently replaced by different random samples.

In [3]:
records = [json.loads(p.read_text()) for p in sorted((RUN / 'answers' / GROUP).glob('*.json'))]
fields = ['dataset', 'file', 'native label', 'prediction', 'score', 'exposure/status']
rows = []
for r in records:
    src = r['input']
    row = [src['dataset'], src['path'], str(src.get('label')), r.get('prediction', r.get('error')), str(round(r['confidence'], 4)) if 'confidence' in r else '', r.get('exposure', r['status'])]
    rows.append('<tr>' + ''.join('<td>' + html.escape(str(v)) + '</td>' for v in row) + '</tr>')
display(HTML('<table><tr>' + ''.join('<th>' + f + '</th>' for f in fields) + '</tr>' + ''.join(rows) + '</table>'))

dataset,file,native label,prediction,score,exposure/status
CUVA,/Datasets/CUVA/raw/group_1/00475.mp4,None,Assault,0.2578,held_out
RWF-2000,/Datasets/RWF-2000/train/NonFight/d3efcb6874a53c53a8f2c93eaa1a38b9ee6b47d4d0c9044ce4f4551bc0451c90.avi,NonFight,Normal_Videos_event,0.5113,held_out
XD-Violence,/Datasets/XD-Violence/train/Fighting/Fighting_v=SE5fxK7SasU___00-04-28_00-04-55_label_B1-0-0.mp4,Fighting,Assault,0.2714,held_out
CUHK Avenue,/Datasets/CUHK Avenue/Avenue Dataset/testing_videos/21.avi,None,Normal_Videos_event,0.6396,held_out
AVA v2.2,/Datasets/AVA v2.2/raw/trainval/_145Aa_xkuE.mp4,None,Normal_Videos_event,0.204,held_out
CUHK Avenue,/Datasets/CUHK Avenue/Avenue Dataset/testing_videos/04.avi,None,Normal_Videos_event,0.6321,held_out
CUHK Avenue,/Datasets/CUHK Avenue/Avenue Dataset/testing_videos/11.avi,None,Normal_Videos_event,0.6306,held_out
CUVA,/Datasets/CUVA/raw/group_2/00611.mp4,None,RoadAccidents,0.2716,held_out
AVA v2.2,/Datasets/AVA v2.2/raw/trainval/uzPI7FcF79U.mkv,None,Abuse,0.2898,held_out
AVA v2.2,/Datasets/AVA v2.2/raw/trainval/jBs_XYHI7gM.mkv,None,Shoplifting,0.2586,held_out


## Limits

UCF training overlap and exact-content aliases affect which folds may score a video. All-video results are descriptive and do not reopen the closed classifier search. The selected external pools are local mirrors, not necessarily complete upstream releases. Different native tasks cannot be ranked by one accuracy value. See config.json, cached_parity.json, attempts.jsonl and the companion notebook for the other group.